In [2]:
import geopandas as gpd
from pathlib import Path

PROJECT_ROOT = next(
    root for root in [Path.cwd(), *Path.cwd().parents]
    if (root / "Data").is_dir() and (root / "pyproject.toml").exists()
)

flow_gdf = gpd.read_file(PROJECT_ROOT / "Data" / "raw" / "structured" / "Port_flows" / "port_flows.shp")

In [3]:
flow_gdf.shape

(20554, 6)

In [4]:
flow_gdf.columns.tolist()
flow_gdf.head()

,Port1,Port2,FLOWS,Country1,Country2,geometry
0,AEAJM,AEJEA,4.075500e+04,AE,AE,"LINESTRING (55.47878 25.40177, 55.10811 25.00255)"
1,AEAUH,AEJEA,1.394829e+04,AE,AE,"LINESTRING (54.36667 24.46667, 55.10811 25.00255)"
2,AEKHL,AEJEA,1.682015e+06,AE,AE,"LINESTRING (54.66667 24.83333, 55.10811 25.00255)"
3,AEKLF,AEJEA,9.186450e+04,AE,AE,"LINESTRING (56.35 25.33333, 55.10811 25.00255)"
4,AEMKH,AEJEA,1.452782e+05,AE,AE,"LINESTRING (55.36667 25.35, 55.10811 25.00255)"


In [6]:
flow_gdf.dtypes


Port1            str
Port2            str
FLOWS        float64
Country1         str
Country2         str
geometry    geometry
dtype: object

In [7]:
flow_gdf.isna().sum()


Port1       0
Port2       0
FLOWS       0
Country1    0
Country2    0
geometry    0
dtype: int64

In [8]:
flow_gdf["geometry"].geom_type.value_counts()

LineString    20554
Name: count, dtype: int64

In [9]:
flow_gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [10]:
flow_gdf[["Port1", "Port2"]].head()

,Port1,Port2
0,AEAJM,AEJEA
1,AEAUH,AEJEA
2,AEKHL,AEJEA
3,AEKLF,AEJEA
4,AEMKH,AEJEA


In [11]:
flow_gdf[["Port1", "Port2"]].duplicated().sum()

np.int64(564)

In [13]:
dup_pairs = flow_gdf[
    flow_gdf[["Port1", "Port2"]].duplicated(keep=False)
]

dup_pairs.groupby(["Port1", "Port2"]).agg(
    rows=("FLOWS", "size"),
    unique_flows=("FLOWS", "nunique"),
    unique_geometry=("geometry", "nunique"),
).sort_values(
    ["unique_flows", "unique_geometry"],
    ascending=False
).head(20)

rows  unique_flows  unique_geometry
Port1 Port2                                     
AGSJO SRPBM     2             1                1
AIAXA TTPOS     2             1                1
ALDRZ DZALG     2             1                1
      ESCAS     2             1                1
      GRPIR     2             1                1
      HRPLE     2             1                1
      HRRJK     2             1                1
      HRSPU     2             1                1
      ITBRI     2             1                1
      ITGIT     2             1                1
      LYMRA     2             1                1
      MEBAR     2             1                1
      SIKOP     2             1                1
      TRALI     2             1                1
      TREYP     2             1                1
      TRMAD     2             1                1
ASPPG NCNOU     2             1                1
AUBNE NCNOU     2             1                1
AUMEL NCNOU     2             1                1
AUPBT NCNOU     2             1                1

In [14]:
flow_gdf = flow_gdf.drop_duplicates().reset_index(drop=True)
flow_gdf.shape

(19990, 6)

In [17]:
%pip install "psycopg[binary]"

   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.6 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.6 MB 940.4 kB/s eta 0:00:04
   -------- ------------------------------- 0.8/3.6 MB 985.6 kB/s eta 0:00:03
   ----------- ---------------------------- 1.0/3.6 MB 1.1 MB/s eta 0:00:03
   ----------- ---------------------------- 1.0/3.6 MB 1.1 MB/s eta 0:00:03
   -------------- ------------------------- 1.3/3.6 MB 979.8 kB/s eta 0:00:03
   -------------- ------------------------- 1.3/3.6 MB 979.8 kB/s eta 0:00:03
   ----------------- ---------------------- 1.6/3.6 MB 920.2 kB/s eta 0:00:03
   -------------------- ------------------- 1.8/3.6 MB 884.3 kB/s eta 0:00:03
   -------------------- ------------------- 1.8/3.6 MB 884.3 kB/s eta 0:00:03
   ---------------------- ----------------- 2.1/3.6 MB 887.1 kB/s eta 0:00:02
   --------------


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import psycopg
import os
from pathlib import Path


def load_env_file(filename=".env"):
    """Load KEY=VALUE lines from the project's .env into os.environ (existing variables win)."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_path = folder / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ.setdefault(key.strip(), value.strip())
            return env_path
    raise FileNotFoundError(f"{filename} not found in {Path.cwd()} or its parents")


load_env_file()


def connect_db():
    """Open a psycopg connection using the PG* variables from .env."""
    return psycopg.connect(
        host=os.environ["PGHOST"],
        port=os.environ["PGPORT"],
        dbname=os.environ["PGDATABASE"],
        user=os.environ["PGUSER"],
        password=os.environ["PGPASSWORD"],
    )

In [19]:
conn = connect_db()

ports_db = set(
    row[0]
    for row in conn.execute("SELECT locode FROM ports")
)

In [20]:
len(ports_db)

837

In [21]:
invalid_port1 = set(flow_gdf["Port1"]) - ports_db
invalid_port2 = set(flow_gdf["Port2"]) - ports_db

len(invalid_port1), len(invalid_port2)

(0, 0)

In [22]:
flow_gdf["FLOWS"].describe()

count    1.999000e+04
mean     1.125384e+05
std      2.397377e+05
min      1.800000e+02
25%      1.295450e+04
50%      4.441956e+04
75%      1.173467e+05
max      6.154786e+06
Name: FLOWS, dtype: float64

In [23]:
flow_gdf["FLOWS"].min(), flow_gdf["FLOWS"].max()

(np.float64(180.0), np.float64(6154785.89))

In [24]:
flow_gdf = flow_gdf.rename(columns={
    "Port1": "port1_locode",
    "Port2": "port2_locode",
    "FLOWS": "flows",
    "Country1": "country1",
    "Country2": "country2",
    "geometry": "geom"
})

In [25]:
flow_gdf.columns.tolist()

['port1_locode', 'port2_locode', 'flows', 'country1', 'country2', 'geom']

In [26]:
conn = connect_db()

In [27]:
with conn.cursor() as cur:
    for row in flow_gdf.itertuples(index=False):
        cur.execute(
            """
            INSERT INTO port_connections (
                port1_locode,
                port2_locode,
                flows,
                country1,
                country2,
                geom
            )
            VALUES (
                %s, %s, %s, %s, %s,
                ST_SetSRID(ST_GeomFromText(%s), 4326)
            )
            """,
            (
                row.port1_locode,
                row.port2_locode,
                row.flows,
                row.country1,
                row.country2,
                row.geom.wkt
            )
        )

conn.commit()

In [28]:
conn.close()

In [29]:
flow_gdf.drop(columns="geom").to_csv(
    PROJECT_ROOT / "Data" / "processed" / "structured" / "port_connections.csv",
    index=False
)
